In [4]:
import getpass
import os

os.environ["DEEPSEEK_API_KEY"] = getpass.getpass()

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model = "deepseek-flash")

 ········


In [5]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi! I'm Hiki")])

AIMessage(content="Hi Hiki! Nice to meet you. 😊 I'm an AI assistant here to help. What would you like to talk about or work on today?", additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer to user. They just said "Hi! I\'m Hiki". Need respond friendly. Need maybe introduce ourselves? As AI assistant. We can say Hi Hiki, nice to meet you. How can I help? Keep concise. But maybe there\'s hidden context? We need answer. Ensure no analysis. Final.\n\nNeed maybe mention "I\'m ChatGPT" or "I\'m an AI assistant". Could ask what they\'d like to do. Since user gave name. Respond warmly. Could say "Hi Hiki! Nice to meet you. I\'m an AI assistant here to help. What would you like to talk about or work on today?" That\'s good.'}, response_metadata={'token_usage': {'completion_tokens': 170, 'prompt_tokens': 36, 'total_tokens': 206, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 137, 'rejected_prediction_tokens': None, 

In [7]:
model.invoke([HumanMessage(content="What's my name?")])
# 我们可以看到它没有将之前的对话轮次作为上下文
# 因此无法回答该问题。 这会导致糟糕的聊天机器人体验!

AIMessage(content='I don’t know your name — you haven’t told me yet. If you share it, I’ll use it.', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer. User asks "What\'s my name?" We don\'t know. Need maybe ask them. We should be honest. Could mention I don\'t have access to personal info unless provided. We can say I don\'t know your name; you haven\'t told me. If you\'d like, tell me and I\'ll use it. Need concise.'}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 35, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 72, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_finger

In [8]:
# 为了绕过这个问题，我们需要将整个对话历史传递给模型
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi! I'm Hiki"),
        AIMessage(content="Hi Hiki! Nice to meet you. How can I help you today?"),
        HumanMessage(content="What's my name?"),
    ]
)
# 这是支撑聊天机器人进行对话交互的基本理念

AIMessage(content='Your name is Hiki.', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer. User says "Hi! I\'m Hiki" then asks "What\'s my name?" We should answer Hiki. Simple. Need maybe "Your name is Hiki." Ensure no confusion.'}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 61, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 43, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 61}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '0cb7ce05-2141-47d8-b77e-6380833ecf46', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c7ba-1f6c-7bd1-b9d8-011072186c0e-0', tool_c

In [9]:
# 消息历史
## 我们可以使用消息历史类来包装我们的模型，使其具有状态
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 它返回一个新的对象 with_message_history，
# 这个新对象和原来的 model 用法几乎一样（也能 .invoke() / .stream()），
# 但多了一层自动管理历史的逻辑
with_message_history = RunnableWithMessageHistory(model, get_session_history)

C:\Users\linxi\Desktop\LangChain\LangChain-Learning\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
config = {"configurable": {"session_id": "abc2"}}

In [14]:
# 把HumanMessage和session_id一起给with_message_history
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Hiki")],
    config=config,
)

response.content

'Hi Hiki! Nice to meet you again. What would you like to chat about?'

In [16]:
# 再次发送同一个session_id
# with_message_history就会先看store中有没有session_id?
# 结果发现已经有了，于是它读取store中的历史并和新message一起发给model
# 然后把新的历史存入store
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Hiki.'

In [19]:
# 如果更改配置，用不同的session_id，它就会开始新对话
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage("What's my name?")],
    config=config,
)

response.content

'I still don’t know—you haven’t told me your name. If you tell me, I can call you that.'

In [21]:
# Meanwhile,我们可以回到之前的对话
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)
response.content

'Your name is Hiki.'

In [22]:
# 提示词模板
## 首先，让我们添加一个系统消息
## 为此，我们将创建一个 ChatPromptTemplate
## 我们将利用 MessagesPlaceholder 来传递所有消息
from langchain_core.prompts import ChatPromptTemplate, MessagePlaceholder

prompt = ChatPromptTemplate.from_message(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability.",
        ),
        MessagePlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

ImportError: cannot import name 'MessagePlaceholder' from 'langchain_core.prompts' (C:\Users\linxi\Desktop\LangChain\LangChain-Learning\.venv\Lib\site-packages\langchain_core\prompts\__init__.py)